In [1]:
try:
    with open("smartchoice_api_key.txt", "r") as file:
        api_key = file.read().strip() 
except FileNotFoundError:
    print("smartchoice_api_key.txt 확인!!")
    exit()

In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

BASE_URL = "https://api.smartchoice.or.kr/api/openAPI.xml"

params = {
    "authkey": api_key,       # 필수: 인증키
    "voice": "999999",                # 필수: 월 평균 통화량(분), 무제한=999999
    "data": "999999",                 # 필수: 월 평균 데이터 사용량(MB), 무제한=999999
    "sms": "999999",                   # 필수: 월 평균 문자 발송량(건), 무제한=999999
    "age": "20",                   # 필수: 연령(성인=20, 청소년=18, 실버=65) -> 20 30 40 결과 같음. 세부 연령대로 구분 불가 
    "type": "6",                   # 필수: 서비스 타입(3G=2, LTE=3, 5G=6)
    "dis": "24",                   # 필수: 약정기간(무약정=0, 12개월=12, 24개월=24)
}

RESULT_CODE_MAP = {
    "001": "인증키 없음",
    "002": "일일 호출 한도(10,000회) 초과",
    "099": "기타 오류",
    "100": "성공",
}

In [9]:
res = requests.get(BASE_URL, params=params)
soup = BeautifulSoup(res.text, "xml")

result_count = soup.find("result_count").text
result_date = soup.find("result_date").text

In [10]:
items = []
for item in soup.find_all("item"):
    items.append({
        "tel": item.find("v_tel").text,
        "plan_name": item.find("v_plan_name").text,
        "plan_price": item.find("v_plan_price").text,
        "dis_price": item.find("v_dis_price").text,
        "plan_over": item.find("v_plan_over").text,
        "add_name": item.find("v_add_name").text.strip(),
        "display_voice": item.find("v_plan_display_voice").text,
        "display_data": item.find("v_plan_display_data").text,
        "display_sms": item.find("v_plan_display_sms").text,
        "rank": item.find("rn").text,
    })

df = pd.DataFrame(items)

In [11]:
condition_labels = {
    "voice": "월 평균 통화량(분)",
    "data": "월 평균 데이터(MB)",
    "sms": "월 평균 문자(건)",
    "age": "연령",
    "type": "서비스 타입",
    "dis": "약정기간(개월)",
}

column_labels = {
    "tel": "통신사",
    "plan_name": "요금제명",
    "plan_price": "요금제 가격",
    "dis_price": "약정할인금액",
    "plan_over": "초과 사용량/금액",
    "add_name": "부가서비스",
    "display_voice": "음성 제공량",
    "display_data": "데이터 제공량",
    "display_sms": "문자 제공량",
    "rank": "추천 등급",
}

df_kr = df.rename(columns=column_labels)


type_map = {"2": "3G", "3": "LTE", "6": "5G"}
dis_map = {"0": "무약정", "12": "12개월", "24": "24개월"}

print("=== 조회 조건 ===")
for key, label in condition_labels.items():
    value = params[key]
    if key == "type":
        value = type_map.get(value, value)
    elif key == "dis":
        value = dis_map.get(value, value)
    print(f"{label}: {value}")

print(f"\n=== 추천 결과 ({result_count}건, {result_date}) ===")
df_kr

=== 조회 조건 ===
월 평균 통화량(분): 999999
월 평균 데이터(MB): 999999
월 평균 문자(건): 999999
연령: 20
서비스 타입: 5G
약정기간(개월): 24개월

=== 추천 결과 (9건, 2026-07-20 16:33:09) ===


,통신사,요금제명,요금제 가격,약정할인금액,초과 사용량/금액,부가서비스,음성 제공량,데이터 제공량,문자 제공량,추천 등급
0,KT,베이직80 Y덤,80000,0,null,"80,000원",유무선 기본 + 영상/부가 300분,무제한,기본제공,1
1,LGU+,데이터플랜MAX,85000,0,null,"85,000원",유무선 기본 + 영상/부가 300분,무제한,기본제공,1
2,SKT,베스트 89,89000,0,null,"89,000원",유무선 기본 + 영상/부가 300분,무제한,기본제공,1
3,SKT,라이트 79 (청년),79000,0,null,"79,000원",유무선 기본 + 영상/부가 300분,300GB + 5Mbps 속도제어,기본제공,2
4,KT,베이직110GB Y덤,69000,0,null,"69,000원",유무선 기본 + 영상/부가 300분,220GB + 5Mbps 속도제어,기본제공,2
5,LGU+,데이터플랜150GB(유쓰+60GB),75000,0,null,"75,000원",유무선 기본 + 영상/부가 300분,210GB + 5Mbps 속도제어,기본제공,2
6,KT,초이스90 Y덤,90000,0,null,"90,000원",유무선 기본 + 영상/부가 300분,무제한,기본제공,3
7,LGU+,플러스플랜95,95000,0,null,"95,000원",유무선 기본 + 영상/부가 300분,무제한,기본제공,3
8,SKT,베스트 99,99000,0,null,"99,000원",유무선 기본 + 영상/부가 300분,무제한,기본제공,3
